# Track B — Safety Net: Embedding-First Pipeline

**BDC Satria Data 2026** | SigLIP2 embedding (beku) → linear probe → submission

> **Pivot:** jalur fine-tuning ConvNeXt (notebook `02_fase1_3_training.ipynb`) dihentikan.
> `hp_results.csv` sudah menunjukkan augmentasi/epoch tidak menolong signifikan.
> Pipeline ini gantinya: ekstrak embedding sekali (beku, tidak di-finetune), lalu
> latih head murah (linear probe) di atasnya.

> **Urutan:** jalankan cell dari atas ke bawah. JANGAN skip cell 3 (cek pola nama
> file test) — `safety_net.py` menebak polanya, harus diverifikasi dulu.

---
## 🔧 SETUP

In [ ]:
# Cell 1 — Clone / update repo
import os
if not os.path.exists('/content/satria-data-bdcugm02'):
    !git clone https://github.com/agaggigit/satria-data-bdcugm02.git
else:
    !git -C /content/satria-data-bdcugm02 pull
print('✅ Repo siap')

In [ ]:
# Cell 2 -- Install dependencies
!pip install -q transformers scikit-learn lightgbm
# hf_xet (HF Hub's newer download backend) currently 401s on anonymous
# requests for some files (tokenizer.json, model.safetensors) -- reproduced
# and confirmed 2026-07-14, matches public reports on HF forums/GitHub.
# Uninstalling it forces the plain-HTTP fallback, which works reliably.
# Do NOT paste an HF_TOKEN directly in a notebook cell in this repo --
# it is public on GitHub. If you ever need one, use Colab Secrets (key icon
# in the left sidebar) instead.
!pip uninstall -y hf_xet hf-xet
print("✅ transformers + scikit-learn + lightgbm siap (hf_xet dilepas)")

In [ ]:
# Cell 3 — Mount Drive + verifikasi artefak
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

import os
for p in [
    '/content/drive/MyDrive/BDC2026apace/output_trackA/folds.csv',
    '/content/drive/MyDrive/BDC2026apace/submission.csv',
    '/content/drive/MyDrive/BDC2026apace/test',
]:
    status = '✅' if os.path.exists(p) else '❌ TIDAK ADA'
    print(status, p)

os.makedirs('/content/drive/MyDrive/BDC2026apace/output_trackB/embeddings', exist_ok=True)
print('✅ output_trackB/embeddings siap (di Drive, bukan runtime lokal)')

In [ ]:
# Cell 4 — ⚠️ WAJIB: cek pola nama file test SEBELUM lanjut
# safety_net.py menebak pola f"{CFG.test_dir}/{id}.jpg" -- ini HARUS diverifikasi
# dulu terhadap isi folder test sungguhan (ekstensi bisa beda, bisa zero-padded, dll).
import os
import pandas as pd

test_dir = '/content/drive/MyDrive/BDC2026apace/test'
sample_files = sorted(os.listdir(test_dir))[:10]
print('Contoh 10 nama file di folder test:')
for f in sample_files:
    print(' ', f)

sub = pd.read_csv('/content/drive/MyDrive/BDC2026apace/submission.csv')
print('\nContoh 5 nilai kolom id di submission.csv:')
print(sub['id'].head().tolist())

print('\n⚠️  Bandingkan manual: apakah pola f"{id}.jpg" di safety_net.py cocok dengan')
print('    nama file di atas? Kalau TIDAK (mis. ekstensi beda / zero-padded),')
print('    edit baris test_paths di track_b/experiments/safety_net.py dulu sebelum lanjut.')

---
## 🚀 SAFETY NET — embedding → linear probe → submission

In [ ]:
# Cell 5 — Jalankan safety_net.py (GPU, ~1 jam total)
# Angka yang ditunggu: CV linear probe di folds.csv. Kirim oof_probe.npy ke Track A
# SEGERA setelah cell ini selesai -- jangan tunggu submission jadi dulu.
import os
os.chdir('/content/satria-data-bdcugm02/track_b/src')
!python ../experiments/safety_net.py

In [ ]:
# Cell 6 — Verifikasi output ada sebelum diumumkan/diunggah
import os
for p in ['oof_probe.npy', 'submission_apace.csv']:
    full = os.path.join('/content/satria-data-bdcugm02/track_b/src', p)
    print(('✅' if os.path.exists(full) else '❌ TIDAK ADA'), full)
print('\n✅ Kalau semua ada: kirim oof_probe.npy ke Track A (GATE B→A),')
print('   lalu unggah submission_apace.csv (catat 1/3, umumkan G-SUB1 hijau).')